##**AMALITECH PROJECT CHALLENGE**

Name:Marie Diane Iradukunda

Email:mariediane.iradukunda@aims.ac.rw




## **Project Brief: The "Sugar Trap" Market Gap Analysis**

## Story 1: Data Ingestion & The Clean Up

Rather than downloading the full ~3GB Open Food Facts export, this streams the
compressed `.csv.gz` file directly and filters for usable rows as it goes,
stopping once a target of ~150,000 usable rows is reached. Rows missing
`product_name`, `sugars_100g`, `proteins_100g`, or `categories_tags` are
dropped rather than imputed, since fabricating central nutrition values would
bias the sugar/protein gap analysis this project is built on. Biologically
impossible values (any nutrient outside 0–100g per 100g) and "empty shell"
entries (0.0 sugar + protein + fat simultaneously) are removed, followed by deduplication.

In [4]:

# STORY 1: DATA INGESTION & THE CLEAN UP

import pandas as pd

# 1. INGESTION: Stream directly from the compressed source URL
URL = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"

COLS_NEEDED = [
    "product_name", "categories_tags", "pnns_groups_1", "pnns_groups_2",
    "brands", "countries_tags",
    "sugars_100g", "proteins_100g", "fat_100g", "fiber_100g",
    "ingredients_text", "nutriscore_grade",
]

ESSENTIAL_COLS = ["product_name", "sugars_100g", "proteins_100g", "categories_tags"]

TARGET_USABLE_ROWS = 150_000
CHUNK_SIZE = 50_000

usable_chunks = []
usable_count = 0
raw_scanned = 0

reader = pd.read_csv(
    URL, sep="\t", compression="gzip", usecols=COLS_NEEDED,
    chunksize=CHUNK_SIZE, on_bad_lines="skip", low_memory=False,
)

for chunk in reader:
    raw_scanned += len(chunk)
    filtered = chunk.dropna(subset=ESSENTIAL_COLS)
    usable_chunks.append(filtered)
    usable_count += len(filtered)
    if usable_count >= TARGET_USABLE_ROWS:
        break

df = pd.concat(usable_chunks, ignore_index=True)
print(f"Raw rows scanned: {raw_scanned:,}")
print(f"Rows with valid product_name, sugars_100g, proteins_100g, categories_tags: {len(df):,}")

# 2. OUTLIER REMOVAL: biologically impossible values
NUTRIENT_COLS = ["sugars_100g", "proteins_100g", "fat_100g", "fiber_100g"]
before_outliers = len(df)

for col in NUTRIENT_COLS:
    df = df[(df[col].isna()) | df[col].between(0, 100)]

# Also drop "empty shell" entries: rows with 0.0 sugar, protein, AND fat

df = df[~((df["sugars_100g"] == 0) & (df["proteins_100g"] == 0) & (df["fat_100g"] == 0))]

print(f"Rows dropped as impossible/placeholder values: {before_outliers - len(df):,}")
print(f"Rows remaining: {len(df):,}")

# 3. DEDUPLICATION
before_dupes = len(df)
df = df.drop_duplicates(subset=["product_name", "brands", "categories_tags"])
print(f"Duplicate rows removed: {before_dupes - len(df):,}")
print(f"Rows remaining: {len(df):,}")

# 4. FINAL SUMMARY (for README Executive Summary)
print("\n--- Cleaning funnel summary ---")
print(f"Raw rows scanned:              {raw_scanned:,}")
print(f"After missing-value filter:    {usable_count:,}")
print(f"After outlier removal:         {before_dupes:,}")
print(f"After deduplication (FINAL):   {len(df):,}")
print(f"Overall retention rate: {len(df)/raw_scanned*100:.2f}%")

# 5. EXPORT — Story 1 deliverable
df.to_csv("cleaned_snacks_data.csv", index=False)
print(f"\nSaved {len(df):,} cleaned rows to cleaned_snacks_data.csv")

df.head()


Raw rows scanned: 1,400,000
Rows with valid product_name, sugars_100g, proteins_100g, categories_tags: 154,839
Rows dropped as impossible/placeholder values: 3,659
Rows remaining: 151,180
Duplicate rows removed: 16,962
Rows remaining: 134,218

--- Cleaning funnel summary ---
Raw rows scanned:              1,400,000
After missing-value filter:    154,839
After outlier removal:         151,180
After deduplication (FINAL):   134,218
Overall retention rate: 9.59%

Saved 134,218 cleaned rows to cleaned_snacks_data.csv


,product_name,brands,categories_tags,countries_tags,ingredients_text,nutriscore_grade,pnns_groups_1,pnns_groups_2,fat_100g,sugars_100g,fiber_100g,proteins_100g
0,Pinto Bean,Central Bean,en:asian-style-ready-meal,"en:united-kingdom,en:world",NaN,unknown,unknown,unknown,10.2,4.9,3.8,17.5
1,pasta,Grappa,"en:beverages-and-beverages-preparations,en:bev...","en:france,en:germany",NaN,c,unknown,unknown,6.4,1.9,0.8,6.7
2,Eirn original curry Sauce,Eirn,"en:plant-based-foods-and-beverages,en:plant-ba...",en:ireland,Wheat Flour Sugar Vegetable Fat D Curry Powder...,not-applicable,unknown,unknown,18.0,29.6,1.2,5.2
3,Donut Milka,Milka,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",en:france,NaN,e,Sugary snacks,Biscuits and cakes,28.0,18.0,NaN,6.0
4,Véritable pâte à tartiner noisettes chocolat noir,Bovetti,"en:breakfasts,en:spreads,en:sweet-spreads,fr:p...",en:france,NaN,e,Sugary snacks,Sweets,48.0,32.0,NaN,8.0


Each criterion above is addressed directly: missing values are filtered during
ingestion (not after), outliers are removed using both a numeric range check
and a check for incomplete "empty shell" entries, and the cleaned result is
exported as cleaned_snacks_data.csv.


## **Story 2: The Category Wrangler**
Before deciding how to group products, I wanted to actually see what tags
show up most often instead of guessing. `categories_tags` has over 15,000
unique values, so most of them are near-useless one-offs the real
signal is in whichever tags appear thousands of times. This just counts
every tag across the cleaned dataset and prints the top 40, which is what
the category rules in the next step are actually based on.

In [5]:

from collections import Counter

df = pd.read_csv("cleaned_snacks_data.csv")

# categories_tags is a comma-separated string like "en:snacks,en:sweet-snacks,en:biscuits"
# Split and count every individual tag across the whole dataset
all_tags = df["categories_tags"].dropna().str.split(",")
tag_counter = Counter()
for tags in all_tags:
    tag_counter.update(t.strip() for t in tags)

print(f"Total unique tags: {len(tag_counter):,}\n")
print("Top 40 most common tags:")
for tag, count in tag_counter.most_common(40):
    print(f"  {tag}: {count:,}")

Total unique tags: 15,431

Top 40 most common tags:
  en:plant-based-foods-and-beverages: 46,374
  en:plant-based-foods: 42,664
  en:snacks: 24,997
  en:cereals-and-potatoes: 23,057
  en:sweet-snacks: 18,523
  en:dairies: 17,110
  en:fermented-foods: 15,015
  en:fermented-milk-products: 14,608
  en:meats-and-their-products: 14,060
  en:breads: 12,213
  en:meats: 10,940
  en:desserts: 10,465
  en:cereals-and-their-products: 10,189
  en:beverages: 8,446
  en:fruits-and-vegetables-based-foods: 8,244
  en:meals: 8,201
  en:biscuits-and-cakes: 8,179
  en:cheeses: 7,699
  en:condiments: 7,417
  en:dairy-desserts: 7,129
  en:breakfasts: 6,841
  en:fermented-dairy-desserts: 6,690
  en:yogurts: 6,364
  en:sauces: 6,041
  en:prepared-meats: 5,502
  en:spreads: 5,240
  en:confectioneries: 5,222
  en:frozen-foods: 4,927
  en:salty-snacks: 4,480
  en:breakfast-cereals: 4,444
  en:vegetables-based-foods: 4,235
  en:plant-based-beverages: 4,030
  en:appetizers: 3,973
  en:poultries: 3,967
  en:biscui

A few clear snack categories stood out from the top tags biscuits,
chocolate, salty snacks, bars, and sweet desserts so the rules are built
around those. Order matters here: rules are checked top to bottom, and the
first match wins. A chocolate-covered biscuit has both tags, but I want it
counted as a biscuit, so that rule runs first. Anything left over is
Non-Snack / Other.

In [8]:

# STORY 2: THE CATEGORY WRANGLER

CATEGORY_RULES = [
    ("Biscuits & Cookies",        ["biscuits-and-cakes", "biscuits", "cookies"]),
    ("Chocolate & Confectionery", ["confectioneries", "cocoa-and-its-products", "chocolates", "candies"]),
    ("Salty & Savory Snacks",     ["salty-snacks", "appetizers", "crisps", "chips",
                                    "popcorn", "crackers", "nuts", "nuts-and-their-products"]),
    ("Bars & Cereal Snacks",      ["breakfast-cereals", "cereal-bars", "granola-bars", "protein-bars"]),
    ("Sweet Snacks & Desserts",   ["sweet-snacks", "desserts", "dairy-desserts", "fermented-dairy-desserts"]),
]

NON_SNACK_FALLBACK = "Non-Snack / Other"

def assign_primary_category(category_tags):
    if pd.isna(category_tags):
        return NON_SNACK_FALLBACK

    # Split the comma-separated string into a list of tags
    tags = [tag.strip() for tag in category_tags.split(',')]

    for category_name, keywords in CATEGORY_RULES:
        for keyword in keywords:
            # Check if any tag contains the keyword
            if any(keyword in tag for tag in tags):
                return category_name

    return NON_SNACK_FALLBACK

df["primary_category"] = df["categories_tags"].apply(assign_primary_category)

print(df["primary_category"].value_counts())
snack_rows = df[df["primary_category"] != NON_SNACK_FALLBACK]
print(f"\nSnack-relevant rows: {len(snack_rows):,} out of {len(df):,} "
      f"({len(snack_rows)/len(df)*100:.1f}%)")


primary_category
Non-Snack / Other            92872
Sweet Snacks & Desserts      12360
Biscuits & Cookies            9184
Chocolate & Confectionery     7157
Salty & Savory Snacks         7032
Bars & Cereal Snacks          5613
Name: count, dtype: int64

Snack-relevant rows: 41,346 out of 134,218 (30.8%)


This turns 15,000+ raw tags into 6 categories I can actually scan at a
glance. As a Product Manager, I can now see at a glance that 30% of the
cleaned dataset is snack-relevant, and which of the 5 categories to focus
on without ever touching a messy tag like
`en:chocolate-chip-cookies-with-nuts`.

## **Story 3: The "Nutrient Matrix" Visualization**

This is implemented as an interactive Streamlit dashboard rather than a
static chart in this notebook. The full scatter plot, category filter,
and empty-quadrant analysis are live at the link below.

Live dashboard: https://amalitech-deg-project-based-challenges-v9lvlazrpt65pkffazgjwj.streamlit.app/

## **Story 4: The Recommendation**

The Key Insight box on the dashboard completes this sentence live, using
whichever category currently has the biggest gap: "Based on the data, the
biggest market opportunity is in Chocolate & Confectionery, specifically
targeting products with 10g of protein and less than 5g of sugar." This
updates automatically if the sensitivity thresholds are adjusted.

## **6.Candidate's Choice: "Who Owns This Shelf?"**

Added a brand-concentration check on the dashboard: a nutrition gap alone
doesn't mean an easy market entry if a handful of brands already dominate
that category. This shows how many unique brands compete in the identified
opportunity category and what share the top 3 hold, turning a purely
nutritional finding into a more complete go/no-go signal for R&D.

## **5.Bonus Story: The "Hidden Gem"**

For products in the high-protein cluster, I want to know what's actually
driving that protein not just that the number is high, but where it's
coming from. This scans `ingredients_text` for common protein-source
keywords and counts how often each one shows up, so R&D has something
concrete to replicate rather than just a target number.

In [9]:

# BONUS STORY: THE "HIDDEN GEM"
from collections import Counter

HIGH_PROTEIN_THRESHOLD = 10  # same threshold used in the dashboard

high_protein_df = df[df["proteins_100g"] >= HIGH_PROTEIN_THRESHOLD].copy()
print(f"Products in the high-protein cluster: {len(high_protein_df):,}")

PROTEIN_KEYWORDS = [
    "whey", "soy", "pea protein", "peanut", "casein", "milk protein",
    "egg white", "almond", "collagen", "rice protein", "hemp protein",
    "chickpea", "lentil", "gelatin",
]

def find_protein_sources(ingredients_text):
    """Return every protein-source keyword found in a product's ingredient list."""
    if pd.isna(ingredients_text):
        return []
    text = ingredients_text.lower()
    return [kw for kw in PROTEIN_KEYWORDS if kw in text]

source_counter = Counter()
for text in high_protein_df["ingredients_text"]:
    for source in find_protein_sources(text):
        source_counter[source] += 1

top3 = source_counter.most_common(3)
print("\nTop 3 protein sources driving the high-protein cluster:")
for i, (source, count) in enumerate(top3, 1):
    print(f"{i}. {source.title()} — appears in {count:,} products")

Products in the high-protein cluster: 47,759

Top 3 protein sources driving the high-protein cluster:
1. Soy — appears in 5,372 products
2. Whey — appears in 2,183 products
3. Peanut — appears in 1,930 products


The top three protein sources are soy (5,372 products), whey (2,183),
and peanut (1,930). Soy and peanut are both plant-based and easy to find,
while whey is still the go to dairy option. Good starting point for R&D(Reaserch and Development Team)
to build a new recipe around.